# Mini Pupper 4 — Colab GPU Training Lab
**Companion to the interactive HTML course.** Run cells in order. The notebook intentionally **stops** if the Colab-host driver is below **580** or a CUDA GPU is not assigned, since the pinned upstream GPU environment uses CUDA 13 packages. It does not upgrade your host driver. Runtime storage is temporary; save artifacts to Drive.

Official source: https://github.com/mangdangroboticsclub/mini_pupper_4_sim | docs/INSTALL.md | docs/TRAINING.md.

**Notebook is a runnable workflow, not a guarantee that the current Colab runtime is compatible.**

## 1 — Select a GPU runtime
Runtime → Change runtime type → select GPU. Then execute the cell. If driver < 580, use a compliant NVIDIA Linux VM/workstation instead.

In [ ]:
import subprocess, re, shutil, os, pathlib
if not shutil.which('nvidia-smi'):
    raise RuntimeError('No NVIDIA GPU visible. In Colab, select Runtime > Change runtime type > GPU.')
p = subprocess.run(['nvidia-smi','--query-gpu=name,driver_version,memory.total','--format=csv,noheader'], text=True, capture_output=True)
print(p.stdout or p.stderr)
if p.returncode: raise RuntimeError('nvidia-smi did not run; GPU runtime unavailable.')
version = p.stdout.splitlines()[0].split(',')[1].strip()
major = int(version.split('.')[0])
print('Host NVIDIA driver major:',major)
if major < 580:
    raise RuntimeError(f'Incompatible NVIDIA driver {version}. Upstream requires >=580. Do not run GPU install on this host.')
print('GPU preflight PASS; this does not guarantee dependencies will resolve.')

## 2 — Check filesystem and clone upstream
The canonical README may refer to `mangdang-bam/mp4-sim`; this notebook uses the original GitHub link from the user. Run from an empty `/content` workspace.

In [ ]:
%cd /content
!df -h /content
!git clone https://github.com/mangdangroboticsclub/mini_pupper_4_sim.git
%cd /content/mini_pupper_4_sim
!git rev-parse HEAD
!ls scripts/install docs policies

## 3 — Install the repository GPU backend
The installer pins the project's dependencies and applies its MuJoCo Warp warning patch. This can take substantial disk and time. If the installer errors, do not proceed to training.

In [ ]:
!bash scripts/install/install.sh --mode gpu

In [ ]:
import subprocess
r=subprocess.run(['.venv/bin/python','-c','import torch; print("torch",torch.__version__,"cuda",torch.cuda.is_available()); assert torch.cuda.is_available()'],capture_output=True,text=True)
print(r.stdout,r.stderr)
if r.returncode: raise RuntimeError('GPU Python environment not ready; inspect installation errors.')

## 4 — Verify pretrained policies (no training yet)

In [ ]:
!.venv/bin/python -m mp4_sim.export.verify policies/flat_omni policies/rough_terrain policies/stairs

## 5 — GPU training smoke test
Only 20 iterations and 128 environments: this validates the pipeline but will NOT produce a high-quality new policy. No auto-resume is promised.

In [ ]:
import os
os.environ['WANDB_MODE']='disabled'
!mkdir -p runs/lessons
!.venv/bin/python -m mp4_sim.gpu.train --variant warm_start --init policies/flat_omni --heading loop --foot-softness 0.005 0.02 --friction 0.4 1.3 --vx-max 0.12 --clearance apex --clearance-weight -2 --stumble -0.5 --progress 2 --envs 128 --out runs/lessons --terrain rough --history 10 --iters 20 --seed 4 --tag colab_smoke

## 6 — TensorBoard and results inspection

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/mini_pupper_4_sim/runs/lessons

In [ ]:
!find runs/lessons -maxdepth 3 -type f | head -40

## 7 — OPTIONAL official stage-1 training (long-running)
Run manually when GPU memory and session limits allow. The recipe defaults to 1024 environments; reduce ENVS if memory is constrained. Do not execute simply to test installation.

In [ ]:
# Uncomment to run full stage 1. Stop if Colab's runtime cannot support it.
# !ENVS=256 WANDB_MODE=disabled sh scripts/recipes/stairs.sh 1

## 8 — OPTIONAL official full curriculum (long-running)
The repository documents six stages; do not attempt blindly on a short ephemeral Colab session. On RTX 5080, upstream estimates ~4 hours; runtimes vary.

In [ ]:
# !ENVS=256 WANDB_MODE=disabled sh scripts/recipes/stairs.sh

## 9 — Evaluate one shipped reference policy (GPU required)
These commands calculate quantitative scores; do not confuse an ONNX verification pass with a walking test.

In [ ]:
!mkdir -p runs/eval
!.venv/bin/python -m mp4_sim.gpu.evaluate --policy policies/rough_terrain --terrain rough --levels 3 6 9 --terrain-seed 7 --heading-loop --stop-at-goal --goal 0.65 --envs 128 --seconds 20 --seed 11 --json runs/eval/rough_colab_seed11.json

## 10 — Export an actual checkpoint (only after training creates one)
Never type `<step run>` literally. Select a real path first. The exporter reads its adjacent config.json.

In [ ]:
from pathlib import Path
ckpts=sorted(Path('runs').rglob('model_*.pt'))
print('Checkpoints:',len(ckpts))
for p in ckpts[-10:]: print(p)
# Select a checkpoint manually after reviewing the list:
# ckpt = str(ckpts[-1])
# !.venv/bin/python -m mp4_sim.export.onnx --checkpoint {ckpt} --out runs/export/my_policy
# !.venv/bin/python -m mp4_sim.export.verify runs/export/my_policy

## 11 — Save your work beyond this ephemeral runtime
Zip small logs/configs/evaluation outputs. Full checkpoint files can be large; copy to Google Drive individually if necessary.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/mini_pupper_rl_course
!tar -czf /content/drive/MyDrive/mini_pupper_rl_course/lesson_results.tar.gz runs/lessons runs/eval 2>/dev/null || true
!ls -lh /content/drive/MyDrive/mini_pupper_rl_course/

## 12 — Student questions
1. Why does the rough policy use 450 rather than 45 values? 2. Why do PPO returns and held-out success differ? 3. Which changed setting is easiest to isolate? 4. What happens if deployment orders the servos incorrectly? 5. How do you document GPU and software version for reproduction?